In [2]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 796.7 kB/s eta 0:00:03
   ------------------- -------------------- 1.0/2.1 MB 1.3 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.6 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.5 MB/s  0:00:01

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

process = Path("../Data/Processed")
report = Path("../Reports/Model_Reports")
model_dir = Path("../Models/Churn")

report.mkdir(parents=True,exist_ok=True)
model_dir.mkdir(parents=True,exist_ok=True)

churn_df = pd.read_csv(process / "retail_churn_features.csv")

FEATURES = [
    "Frequency","Monetary","TotalQuantity","AverageOrderValue",
    "CustomerLifetimeDays","purchase_rate","revenue_per_day",
    "qty_per_order","avg_gap","spend_per_visit","is_one_time",
    "qty_per_day","is_slow_buyer","aov_ratio"
]

X = churn_df[FEATURES].copy()
y = churn_df["Churned"].copy()

X = X.replace([np.inf,-np.inf],np.nan).fillna(0)

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,stratify=y,random_state=42
)

print("Training:",X_train.shape)
print("Testing:",X_test.shape)
print("Train churn rate:",round(y_train.mean()*100,2),"%")
print("Test churn rate:",round(y_test.mean()*100,2),"%")

Training: (4702, 14)
Testing: (1176, 14)
Train churn rate: 50.85 %
Test churn rate: 50.85 %


In [2]:
import optuna
import xgboost as xgb
import joblib
from sklearn.metrics import roc_auc_score,classification_report,average_precision_score

X_fit,X_val,y_fit,y_val = train_test_split(
    X_train,y_train,test_size=0.2,stratify=y_train,random_state=42
)

scale = (y_fit == 0).sum()/(y_fit == 1).sum()

print("Fit:",X_fit.shape)
print("Validation:",X_val.shape)
print("Testing:",X_test.shape)
print("scale_pos_weight:",round(scale,2))

Fit: (3761, 14)
Validation: (941, 14)
Testing: (1176, 14)
scale_pos_weight: 0.97


c:\Users\Dharm Maniya\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def objective(trial):
    model = xgb.XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators",300,1000),
        max_depth=trial.suggest_int("max_depth",3,8),
        learning_rate=trial.suggest_float("learning_rate",0.01,0.15,log=True),
        min_child_weight=trial.suggest_int("min_child_weight",1,10),
        subsample=trial.suggest_float("subsample",0.7,1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree",0.7,1.0),
        gamma=trial.suggest_float("gamma",0,1),
        reg_alpha=trial.suggest_float("reg_alpha",0.001,1,log=True),
        reg_lambda=trial.suggest_float("reg_lambda",0.1,10,log=True),
        scale_pos_weight=scale,
        random_state=42,
        eval_metric="auc",
        early_stopping_rounds=30,
        verbosity=0
    )

    model.fit(X_fit,y_fit,eval_set=[(X_val,y_val)],verbose=False)

    y_prob = model.predict_proba(X_val)[:,1]

    return roc_auc_score(y_val,y_prob)

In [4]:
study = optuna.create_study(direction="maximize")

study.optimize(objective,n_trials=40)

print("Best validation AUC:",round(study.best_value,4))
print("\nBest parameters:")
print(study.best_params)

[I 2026-06-12 14:19:12,007] A new study created in memory with name: no-name-8bbcebce-57cb-4ef9-a5f5-5cf6cead922d
[I 2026-06-12 14:19:12,180] Trial 0 finished with value: 0.7940785727842096 and parameters: {'n_estimators': 554, 'max_depth': 3, 'learning_rate': 0.07996187890696833, 'min_child_weight': 3, 'subsample': 0.8243457634945964, 'colsample_bytree': 0.8123600682502428, 'gamma': 0.4485420848661047, 'reg_alpha': 0.039319904867060024, 'reg_lambda': 1.367154713903305}. Best is trial 0 with value: 0.7940785727842096.
[I 2026-06-12 14:19:12,262] Trial 1 finished with value: 0.7906352520131226 and parameters: {'n_estimators': 854, 'max_depth': 8, 'learning_rate': 0.01535701870154045, 'min_child_weight': 9, 'subsample': 0.9123668157368489, 'colsample_bytree': 0.9557391403174473, 'gamma': 0.10260364171844416, 'reg_alpha': 0.0013129257508670998, 'reg_lambda': 0.2967601431002869}. Best is trial 0 with value: 0.7940785727842096.
[I 2026-06-12 14:19:12,357] Trial 2 finished with value: 0.7927

Best validation AUC: 0.7968

Best parameters:
{'n_estimators': 628, 'max_depth': 5, 'learning_rate': 0.01042946099443205, 'min_child_weight': 3, 'subsample': 0.875750061952896, 'colsample_bytree': 0.9977182367976457, 'gamma': 0.43085930743467377, 'reg_alpha': 0.003365104299635311, 'reg_lambda': 0.21952140101191078}


In [5]:
best_params = study.best_params

tuned_model = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric="auc",
    early_stopping_rounds=30,
    verbosity=0
)

tuned_model.fit(
    X_fit,y_fit,
    eval_set=[(X_val,y_val)],
    verbose=False
)

print("Best validation AUC:",round(study.best_value,4))
print("Best iteration:",tuned_model.best_iteration)

Best validation AUC: 0.7968
Best iteration: 8


In [6]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

y_prob_tuned = tuned_model.predict_proba(X_test)[:,1]
y_pred_tuned = tuned_model.predict(X_test)

auc_tuned = roc_auc_score(y_test,y_prob_tuned)
auc_pr_tuned = average_precision_score(y_test,y_prob_tuned)

print("========== TUNED CHURN MODEL RESULTS ==========")
print("AUC-ROC:",round(auc_tuned,4))
print("AUC-PR :",round(auc_pr_tuned,4))
print("\n",classification_report(
    y_test,y_pred_tuned,
    target_names=["Active","Churned"]
))

========== TUNED CHURN MODEL RESULTS ==========
AUC-ROC: 0.8129
AUC-PR : 0.772

               precision    recall  f1-score   support

      Active       0.75      0.68      0.71       578
     Churned       0.71      0.78      0.75       598

    accuracy                           0.73      1176
   macro avg       0.73      0.73      0.73      1176
weighted avg       0.73      0.73      0.73      1176



In [7]:
comparison_df = pd.DataFrame({
    "Model":["XGBoost Baseline","XGBoost Tuned"],
    "AUC_ROC":[0.7971,auc_tuned],
    "AUC_PR":[0.7750,auc_pr_tuned]
}).round(4)

comparison_df

,Model,AUC_ROC,AUC_PR
0,XGBoost Baseline,0.7971,0.775
1,XGBoost Tuned,0.8129,0.772


In [8]:
joblib.dump(
    tuned_model,
    model_dir / "xgboost_churn_tuned.pkl"
)

comparison_df.to_csv(
    report / "11-Churn_Tuning_Comparison.csv",
    index=False
)

print("Tuned churn model saved successfully.")

Tuned churn model saved successfully.


## Churn Model — Target Gap Analysis

The final model achieved an **AUC-ROC score of `0.8129`**, representing a reliable transaction-based churn prediction baseline.

The remaining gap to the target score of **`0.88`** is mainly due to limited data coverage. Further improvement would require additional customer signals such as:

* Engagement activity
* Refund history
* Customer-support interactions
* Satisfaction indicators
